In [ ]:
# ============================================================
# Fix best GO-LR + NSC params from Optuna, vary only tabpfn_seed
# Report:
#   - per-seed 5x5 CV mean ± std accuracy
#   - across-seed summary (mean/std/min/max/range of seed-wise means)
# ============================================================

import os, sys, gc, random, warnings, time
warnings.filterwarnings("ignore")

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config
PIDFSegPCA = pidf_segpca

# -----------------------
# Config
# -----------------------
SEED = 42
DATA_FILE = "coloncancer_encoded.csv"
TARGET_COL = "label"
GPU_ID = 7

# Best fixed params from Optuna
BEST_PARAMS = {
    "go_metric": "euclidean",
    "go_num_clusters": 10,
    "go_refine_passes": 3,
    "go_direction_select": True,
    "nsc_segmentation": "equal_mass",
    "nsc_m_rule": "idf",
    "nsc_tau": 0.99,
    "nsc_gamma": 1.7570143129240916,
    "nsc_beta": 0.2244046472232107,
    "nsc_Mmin": 64,
    "nsc_Mmax": 384,
    "nsc_lmin": 16,
    "assume_standardized": False,
}

# Seed set for sensitivity analysis
TABPFN_SEEDS = [0, 1, 2, 3, 4, 7, 11, 17, 23, 42]

# If you want to force which GPU is visible
if "torch" not in sys.modules:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)

# -----------------------
# Utils
# -----------------------
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()

def ensure_multiclass_contiguous(y: np.ndarray):
    y = np.asarray(y).reshape(-1)
    uniq = np.unique(y)
    uniq_sorted = np.sort(uniq)
    class_map = {orig: i for i, orig in enumerate(uniq_sorted.tolist())}
    y_enc = np.vectorize(class_map.get, otypes=[np.int64])(y).astype(np.int64)
    return y_enc, class_map, int(len(class_map))

def compute_deltas_adjacent_corr(X_tr: np.ndarray, Pi_star: list[int], eps: float = 1e-12) -> torch.Tensor:
    """
    delta[t] = 1 - |corr(feature_{Pi[t]}, feature_{Pi[t+1]})|
    Returns torch.Tensor shape (m-1,) on CPU.
    """
    X = torch.from_numpy(X_tr).float()  # CPU
    perm = torch.tensor(Pi_star, dtype=torch.long)
    Xp = X[:, perm]
    Xc = Xp - Xp.mean(dim=0, keepdim=True)
    std = Xc.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    Z = Xc / std
    corr = (Z[:, :-1] * Z[:, 1:]).mean(dim=0)
    return (1.0 - corr.abs()).cpu()

# -----------------------
# Load data
# -----------------------
seed_everything(SEED)
df = pd.read_csv(DATA_FILE)

y_raw = df[TARGET_COL].to_numpy()
X_df = df.drop(columns=[TARGET_COL])

y_all, class_map, NUM_CLASSES = ensure_multiclass_contiguous(y_raw)

# Match your original pipeline: standardize globally once
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df.values).astype(np.float32, copy=False)

X_all = np.asarray(X_scaled, dtype=np.float32, order="C")
y_all = np.asarray(y_all, dtype=np.int64)

print(f"[DATA] {DATA_FILE} | X={X_all.shape} | C={NUM_CLASSES} | map={class_map}")
print("[GPU ] cuda=", torch.cuda.is_available(), "| visible_gpus=", torch.cuda.device_count())

# Fixed 5x5 CV splits
rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
splits = list(rkf.split(X_all, y_all))

# Devices
NSC_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TABPFN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------
# Fit GO-LR ONCE using fixed best params
# -----------------------
seed_everything(SEED)

go = GraphFeatureOrdering(
    num_clusters=int(BEST_PARAMS["go_num_clusters"]),
    metric=BEST_PARAMS["go_metric"],
    refine=True,
    direction_select=bool(BEST_PARAMS["go_direction_select"]),
    refine_passes=int(BEST_PARAMS["go_refine_passes"]),
)

print("\n[FIT] Building fixed Pi_star with best GO-LR params ...")
try:
    Pi_star, _, _, _ = go.fit(X_all, seed=SEED, deterministic=True, use_cpu_kmeans=False)
except RuntimeError:
    cleanup_cuda()
    Pi_star, _, _, _ = go.fit(X_all, seed=SEED, deterministic=True, use_cpu_kmeans=True)

print(f"[DONE] Pi_star length = {len(Pi_star)}")

# -----------------------
# Evaluate one tabpfn seed
# -----------------------
def evaluate_for_tabpfn_seed(tabpfn_seed: int):
    seed_everything(SEED)  # keep everything else deterministic
    fold_accs = []
    t0 = time.time()

    head_cfg = TabPFN25Config(
        task_type="binary" if NUM_CLASSES == 2 else "multiclass",
        num_classes=int(NUM_CLASSES),
        device=TABPFN_DEVICE,
        random_state=int(tabpfn_seed),
    )

    for fold_id, (tr_idx, va_idx) in enumerate(splits, start=1):
        X_tr = X_all[tr_idx]
        y_tr = y_all[tr_idx]
        X_va = X_all[va_idx]
        y_va = y_all[va_idx]

        # Configure NSC on TRAIN only, using fixed Pi_star
        nsc = PIDFSegPCA(
            segmentation=BEST_PARAMS["nsc_segmentation"],
            l_min=int(BEST_PARAMS["nsc_lmin"]),
            m_rule=BEST_PARAMS["nsc_m_rule"],
            gamma=float(BEST_PARAMS["nsc_gamma"]),
            beta=float(BEST_PARAMS["nsc_beta"]),
            tau=float(BEST_PARAMS["nsc_tau"]),
            M_min=int(BEST_PARAMS["nsc_Mmin"]),
            M_max=int(BEST_PARAMS["nsc_Mmax"]),
            assume_standardized=bool(BEST_PARAMS["assume_standardized"]),
            device=NSC_DEVICE,
        )

        deltas = None
        if BEST_PARAMS["nsc_segmentation"] != "uniform":
            deltas = compute_deltas_adjacent_corr(X_tr, Pi_star)

        X_tr_t = torch.from_numpy(X_tr)
        X_va_t = torch.from_numpy(X_va)

        nsc.configure(
            Pi_star=Pi_star,
            X_train=X_tr_t,
            tau=float(BEST_PARAMS["nsc_tau"]),
            deltas=deltas,
        )

        Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
        Z_va = nsc.compress(X_va_t, mode="flatten").cpu().numpy()

        head = TabPFN25Head(head_cfg)
        head.fit(Z_tr, y_tr)

        P = head.predict_proba(Z_va)
        pred = np.argmax(P, axis=1)
        acc = float(accuracy_score(y_va, pred))
        fold_accs.append(acc)

        cleanup_cuda()

    elapsed = time.time() - t0
    mean_acc = float(np.mean(fold_accs))
    std_acc = float(np.std(fold_accs, ddof=1))
    return {
        "tabpfn_seed": int(tabpfn_seed),
        "mean_acc": mean_acc,
        "std_acc": std_acc,
        "min_fold_acc": float(np.min(fold_accs)),
        "max_fold_acc": float(np.max(fold_accs)),
        "runtime_sec": float(elapsed),
        "fold_accs": fold_accs,
    }

# -----------------------
# Run seed sensitivity
# -----------------------
print("\n[RUN] Seed sensitivity over TabPFN seeds:", TABPFN_SEEDS)

results = []
for s in TABPFN_SEEDS:
    print(f"\n--- Evaluating TabPFN seed = {s} ---")
    out = evaluate_for_tabpfn_seed(s)
    results.append(out)
    print(
        f"seed={s:>2d} | mean_acc={out['mean_acc']:.6f} | "
        f"std_acc={out['std_acc']:.6f} | runtime={out['runtime_sec']:.2f}s"
    )

# -----------------------
# Summaries
# -----------------------
res_df = pd.DataFrame([
    {
        "TabPFN Seed": r["tabpfn_seed"],
        "Mean Accuracy": r["mean_acc"],
        "Std over 25 folds": r["std_acc"],
        "Min Fold Acc": r["min_fold_acc"],
        "Max Fold Acc": r["max_fold_acc"],
        "Runtime (sec)": r["runtime_sec"],
    }
    for r in results
]).sort_values("TabPFN Seed").reset_index(drop=True)

seed_means = res_df["Mean Accuracy"].to_numpy()
summary = {
    "Across-seed mean of 25-fold means": float(np.mean(seed_means)),
    "Across-seed std of 25-fold means": float(np.std(seed_means, ddof=1)) if len(seed_means) > 1 else 0.0,
    "Across-seed min of 25-fold means": float(np.min(seed_means)),
    "Across-seed max of 25-fold means": float(np.max(seed_means)),
    "Across-seed range of 25-fold means": float(np.max(seed_means) - np.min(seed_means)),
    "Best seed by mean accuracy": int(res_df.loc[res_df["Mean Accuracy"].idxmax(), "TabPFN Seed"]),
}

# Pretty print
print("\n" + "=" * 100)
print("Seed Sensitivity Results: GOTabPFN on Colon (best GO-LR + NSC fixed, only TabPFN seed varied)")
print("=" * 100)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 200):
    printable_df = res_df.copy()
    printable_df["Mean Accuracy"] = printable_df["Mean Accuracy"].map(lambda x: f"{x:.4f}")
    printable_df["Std over 25 folds"] = printable_df["Std over 25 folds"].map(lambda x: f"{x:.4f}")
    printable_df["Min Fold Acc"] = printable_df["Min Fold Acc"].map(lambda x: f"{x:.4f}")
    printable_df["Max Fold Acc"] = printable_df["Max Fold Acc"].map(lambda x: f"{x:.4f}")
    printable_df["Runtime (sec)"] = printable_df["Runtime (sec)"].map(lambda x: f"{x:.2f}")
    print(printable_df.to_string(index=False))

print("\nAcross-seed summary")
for k, v in summary.items():
    if isinstance(v, float):
        print(f"{k}: {v:.6f}")
    else:
        print(f"{k}: {v}")

# Optional: CSV outputs for rebuttal table
res_df.to_csv("colon_tabpfn_seed_sensitivity.csv", index=False)

summary_df = pd.DataFrame([summary])
summary_df.to_csv("colon_tabpfn_seed_sensitivity_summary.csv", index=False)

print("\nSaved:")
print("  - colon_tabpfn_seed_sensitivity.csv")
print("  - colon_tabpfn_seed_sensitivity_summary.csv")

[DATA] coloncancer_encoded.csv | X=(62, 2000) | C=2 | map={0: 0, 1: 1}
[GPU ] cuda= True | visible_gpus= 8

[FIT] Building fixed Pi_star with best GO-LR params ...
[DONE] Pi_star length = 2000

[RUN] Seed sensitivity over TabPFN seeds: [0, 1, 2, 3, 4, 7, 11, 17, 23, 42]

--- Evaluating TabPFN seed = 0 ---
seed= 0 | mean_acc=0.875128 | std_acc=0.087211 | runtime=44.56s

--- Evaluating TabPFN seed = 1 ---
seed= 1 | mean_acc=0.865641 | std_acc=0.092459 | runtime=55.45s

--- Evaluating TabPFN seed = 2 ---
seed= 2 | mean_acc=0.871795 | std_acc=0.087150 | runtime=47.47s

--- Evaluating TabPFN seed = 3 ---
seed= 3 | mean_acc=0.868974 | std_acc=0.083376 | runtime=40.75s

--- Evaluating TabPFN seed = 4 ---
seed= 4 | mean_acc=0.868718 | std_acc=0.087113 | runtime=42.27s

--- Evaluating TabPFN seed = 7 ---
seed= 7 | mean_acc=0.868462 | std_acc=0.083561 | runtime=52.22s

--- Evaluating TabPFN seed = 11 ---
seed=11 | mean_acc=0.871538 | std_acc=0.093417 | runtime=45.00s

--- Evaluating TabPFN seed 